# Invoice Region Detection and Business Parameter Extraction Using CNN, SSD, IoU, OCR, and Streamlit

**Member:** Diana
**Role:** Annotation, Stamp Detection, Signature Detection Lead

**Objective:** Detect stamps and signatures on invoice images as two SEPARATE object classes, evaluate with precision/recall/IoU per class, and save predictions + metrics + trained model artifacts.

**Inputs expected:**
- `data/processed/invoice_manifest.csv` (Rolando)
- SignverOD (`data/raw/signatures/`) and StaVer (`data/raw/stamps/`) datasets

**Outputs generated:**
- `outputs/predictions/stamp_signature_predictions.csv`
- `outputs/metrics/stamp_signature_metrics.json`
- `outputs/figures/stamp_signature_detection_examples.png`
- `models/stamp_detector/`, `models/signature_detector/`

> Run this notebook top-to-bottom in Google Colab, or locally with the repo's virtualenv.
> Paths are resolved via `src/config.py` (pathlib-based) — never hardcode absolute local paths.


In [ ]:
# --- Google Colab setup cell ---
# If running in Colab: clone the repo (or mount Drive if you cloned there already) and
# install dependencies. Safe to skip locally if the repo is already on disk with deps installed.

import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/<your-org>/invoice-image-processing.git"  # TODO: set this
    REPO_DIR = "/content/invoice-image-processing"

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q -r requirements.txt")

    # Kaggle API credentials (upload kaggle.json when prompted) -- see dataset_sources.md
    from google.colab import files
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token):")
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        for fname in uploaded:
            os.replace(fname, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    print("Colab environment ready. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- assuming local repo checkout with requirements installed.")


In [ ]:
# --- Dataset path setup cell ---
# All paths go through src.config.PATHS (pathlib-based, no hardcoded absolute paths).

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PATHS, load_label_schema, load_required_fields

print("Repo root:", PATHS.repo_root)
print("Raw data dir:", PATHS.raw_dir)
print("Outputs dir:", PATHS.outputs_dir)

# If raw data isn't present yet, download it (see dataset_sources.md for kaggle.json setup):
#   python scripts/download_datasets.py --dataset all


In [ ]:
# --- Imports cell ---
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.image_preprocessing import preprocess_pipeline, to_grayscale, resize_image, denoise_image, threshold_image, deskew_image
from src.visualization import draw_boxes, show_image_grid
from src.annotation_utils import load_annotations, boxes_for_image
from src.iou import compute_iou, precision_recall_iou, evaluate_predictions_df


## Label rule (do not violate)

`stamp` and `signature` are always kept as two separate labels. Never merge them into a
single "authorization mark" class, and never rename either label.

In [ ]:
from src.stamp_signature_detection import STAMP_LABEL, SIGNATURE_LABEL, VISUAL_ELEMENT_LABELS
print(VISUAL_ELEMENT_LABELS)


## 1. Load SignverOD (signatures) and StaVer (stamps), normalize labels

In [ ]:
# TODO: load SignverOD annotations -> rows with label='signature'
# TODO: load StaVer annotations -> rows with label='stamp'
# Normalize both into the shared schema and write to data/annotations/stamp_signature_bboxes.csv:
#   document_id,image_path,label,xmin,ymin,xmax,ymax,split,annotation_source
from src.annotation_utils import append_annotations

# Example of the target row shape once parsed from the raw dataset formats:
# append_annotations([
#     {"document_id": "sig_0001", "image_path": "data/raw/signatures/sig_0001.png", "label": "signature",
#      "xmin": 120, "ymin": 340, "xmax": 300, "ymax": 410, "split": "train", "annotation_source": "signverod"},
# ], PATHS.annotations_dir / "stamp_signature_bboxes.csv")


## 2. Load merged stamp/signature annotations

In [ ]:
from src.annotation_utils import load_annotations

ann_path = PATHS.annotations_dir / "stamp_signature_bboxes.csv"
if ann_path.exists() and ann_path.stat().st_size > 0:
    stamp_sig_annotations = load_annotations(ann_path, labels=VISUAL_ELEMENT_LABELS)
else:
    stamp_sig_annotations = pd.DataFrame(columns=["document_id","image_path","label","xmin","ymin","xmax","ymax","split","annotation_source"])
stamp_sig_annotations.head()


## 3. Train or demonstrate stamp/signature detection

In [ ]:
# TODO: train (or load a pretrained) detector -- e.g. an ultralytics YOLOv8n model
# fine-tuned on the merged stamp+signature annotation set, OR two separate lightweight
# detectors. Save weights to models/stamp_detector/ and models/signature_detector/.
#
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")
# model.train(data=<yaml pointing at stamp_signature_bboxes.csv-derived YOLO dataset>, epochs=20)


## 4. Run inference, build predictions dataframe

In [ ]:
# TODO: run the trained detector over the manifest's images (or a sample) and collect
# predictions into a dataframe with columns: document_id, image_path, label, xmin, ymin,
# xmax, ymax, confidence.
predictions = pd.DataFrame(columns=["document_id", "image_path", "label", "xmin", "ymin", "xmax", "ymax", "confidence"])


## 5. Evaluate: precision / recall / IoU, separately for stamp and signature

In [ ]:
from src.iou import evaluate_predictions_df

if not predictions.empty and not stamp_sig_annotations.empty:
    metrics = evaluate_predictions_df(predictions, stamp_sig_annotations, VISUAL_ELEMENT_LABELS, iou_threshold=0.5)
else:
    metrics = {"per_label": {label: {"precision": None, "recall": None, "mean_iou": None} for label in VISUAL_ELEMENT_LABELS}, "overall_mean_iou": None}

stamp_signature_metrics = {
    "stamp": metrics["per_label"].get("stamp", {}),
    "signature": metrics["per_label"].get("signature", {}),
}
print(json.dumps(stamp_signature_metrics, indent=2))


## 6. Save example predictions drawn on images

In [ ]:
from src.visualization import draw_boxes, show_image_grid

example_imgs, example_titles = [], []
for doc_id in predictions["document_id"].unique()[:6]:
    row = manifest_row = None  # TODO: look up image path from Rolando's manifest by doc_id
    # img = cv2.imread(...); boxes = predictions[predictions.document_id == doc_id].to_dict("records")
    # example_imgs.append(draw_boxes(img, boxes)); example_titles.append(doc_id)

PATHS.figures_dir.mkdir(parents=True, exist_ok=True)
if example_imgs:
    show_image_grid(example_imgs, titles=example_titles, cols=3, save_path=PATHS.figures_dir / "stamp_signature_detection_examples.png")
else:
    print("No example predictions yet -- fill in section 3/4 with a trained model first.")


In [ ]:
# --- Final export cell ---
# Save every output required by model_interface_contract.md

PATHS.predictions_dir.mkdir(parents=True, exist_ok=True)
predictions.to_csv(PATHS.predictions_dir / "stamp_signature_predictions.csv", index=False)

PATHS.metrics_dir.mkdir(parents=True, exist_ok=True)
(PATHS.metrics_dir / "stamp_signature_metrics.json").write_text(json.dumps(stamp_signature_metrics, indent=2), encoding="utf-8")

(PATHS.models_dir / "stamp_detector").mkdir(parents=True, exist_ok=True)
(PATHS.models_dir / "signature_detector").mkdir(parents=True, exist_ok=True)
# TODO: save actual trained weights into the two folders above.

member_out = PATHS.member_outputs_dir("diana_stamp_signature")
member_out.mkdir(parents=True, exist_ok=True)
predictions.to_csv(member_out / "stamp_signature_predictions.csv", index=False)
(member_out / "stamp_signature_metrics.json").write_text(json.dumps(stamp_signature_metrics, indent=2), encoding="utf-8")

print('Export complete.')
